# Data Preprocessing Practice

In [2]:
# Importing libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

## Step 1: Load and Inspect the Data

- Read the student_data_assignment.csv file using Pandas.

- Separate the dataset into Features (X) and the Target (y).

> Hint: **Name** is just an identifier, you should drop it. **Passed** is your target *(y)*. Everything else is *X*.

In [3]:
# Reading Data
df = pd.read_csv('student_data_assignment.csv')
df

,Name,Age,Gender,City,Score,Passed
0,Aman,22.0,Male,Delhi,85.0,Yes
1,Priya,25.0,Female,Mumbai,90.0,Yes
2,Rahul,NaN,Male,Bangalore,78.0,No
3,Anjali,23.0,Female,Mumbai,NaN,Yes
4,Ravi,28.0,Male,Delhi,88.0,Yes
5,Meera,24.0,Female,Chennai,75.0,No
6,Arjun,26.0,Male,Bangalore,92.0,Yes
7,Neha,NaN,Female,Delhi,85.0,Yes
8,Imran,29.0,Male,Chennai,70.0,No
9,Sneha,22.0,Female,Mumbai,89.0,Yes


In [4]:
# Separate the dataset into Features (X) and the Target (y)
X = df.drop(columns=['Name', 'Passed'])
y = df['Passed']

## Step 2: Handle Missing Data

You will notice missing values in the *Age* and *Score* columns.

- Use SimpleImputer from sklearn.impute to fill the missing values.

- Replace the missing numbers with the mean (average) of their respective columns.

In [5]:
imputer = SimpleImputer(missing_values=np.nan, strategy="mean")
X[['Age','Score']] = imputer.fit_transform(X[['Age','Score']])

In [6]:
X.isnull().sum()

Age       0
Gender    0
City      0
Score     0
dtype: int64

In [7]:
X

,Age,Gender,City,Score
0,22.000000,Male,Delhi,85.000000
1,25.000000,Female,Mumbai,90.000000
2,25.058824,Male,Bangalore,78.000000
3,23.000000,Female,Mumbai,81.941176
4,28.000000,Male,Delhi,88.000000
5,24.000000,Female,Chennai,75.000000
6,26.000000,Male,Bangalore,92.000000
7,25.058824,Female,Delhi,85.000000
8,29.000000,Male,Chennai,70.000000
9,22.000000,Female,Mumbai,89.000000


## Step 3: Encode Categorical Data

Machine learning algorithms only understand numbers.

- Use *OneHotEncoder* from sklearn.preprocessing (or pd.get_dummies) to encode the nominal categorical features: Gender and City.

- Use *LabelEncoder* to encode the target variable: Passed (Yes = 1, No = 0).

In [8]:
# sparse_output=False ensures it returns a readable matrix of 1s and 0s
oh_encoder = OneHotEncoder(sparse_output=False)

# Fit and transform the two nominal columns
encoder_matrix = oh_encoder.fit_transform(df[['Gender','City']])

# Scikit-learn returns a raw Numpy array, so we need to get the new column names...
new_columns_name = oh_encoder.get_feature_names_out(['Gender','City'])

# convert that array back into a Pandas DataFrame
encoded_df = pd.DataFrame(encoder_matrix,columns=new_columns_name)
encoded_df

,Gender_Female,Gender_Male,City_Bangalore,City_Chennai,City_Delhi,City_Mumbai
0,0.0,1.0,0.0,0.0,1.0,0.0
1,1.0,0.0,0.0,0.0,0.0,1.0
2,0.0,1.0,1.0,0.0,0.0,0.0
3,1.0,0.0,0.0,0.0,0.0,1.0
4,0.0,1.0,0.0,0.0,1.0,0.0
5,1.0,0.0,0.0,1.0,0.0,0.0
6,0.0,1.0,1.0,0.0,0.0,0.0
7,1.0,0.0,0.0,0.0,1.0,0.0
8,0.0,1.0,0.0,1.0,0.0,0.0
9,1.0,0.0,0.0,0.0,0.0,1.0


In [9]:
lb_encoder = LabelEncoder()
y_encoded  = lb_encoder.fit_transform(y)
display(y_encoded)

array([1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1])

In [10]:
X = X.drop(columns=['Gender','City'])

In [11]:
X_encoded = pd.concat([encoded_df,X],axis=1)
X_encoded

,Gender_Female,Gender_Male,City_Bangalore,City_Chennai,City_Delhi,City_Mumbai,Age,Score
0,0.0,1.0,0.0,0.0,1.0,0.0,22.000000,85.000000
1,1.0,0.0,0.0,0.0,0.0,1.0,25.000000,90.000000
2,0.0,1.0,1.0,0.0,0.0,0.0,25.058824,78.000000
3,1.0,0.0,0.0,0.0,0.0,1.0,23.000000,81.941176
4,0.0,1.0,0.0,0.0,1.0,0.0,28.000000,88.000000
5,1.0,0.0,0.0,1.0,0.0,0.0,24.000000,75.000000
6,0.0,1.0,1.0,0.0,0.0,0.0,26.000000,92.000000
7,1.0,0.0,0.0,0.0,1.0,0.0,25.058824,85.000000
8,0.0,1.0,0.0,1.0,0.0,0.0,29.000000,70.000000
9,1.0,0.0,0.0,0.0,0.0,1.0,22.000000,89.000000


## Step 4: Train/Test Split
Before scaling, you must split your data to prevent data leakage (where the test set accidentally influences the scaling of the training set).

- Use `train_test_split` from `sklearn.model_selection`.

- Split the data into 80% training data and 20% testing data `(test_size=0.2)`. Assign a `random_state` so your results are reproducible.

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y_encoded, train_size=0.2,random_state=42)

In [13]:
print("Original dataset size:", len(X_encoded), "rows")
print("Training data size:", len(X_train), "rows")
print("Testing data size:", len(X_test), "rows")

Original dataset size: 20 rows
Training data size: 4 rows
Testing data size: 16 rows


## Step 5: Feature Scaling
Now scale the numerical columns (Age and Score). Only fit the scaler on the training data, then transform both train and test data.

Exercise A: Apply `StandardScaler` (Z-score scaling) to your numerical columns.

Exercise B: Apply `MinMaxScaler` (0 to 1 scaling) to your numerical columns. Observe how the arrays differ.

### Exercise A: StandardScaler

In [ ]:
# 1. Create the scaler object
std_scaler = StandardScaler()

# 2. Tell the scaler which columns need to be scaled 
# (We don't want to scale our 1s and 0s from the City/Gender columns!)
col_to_scale = ['Age','Score']

# 3. Fit and Transform the TRAINING data
X_train[col_to_scale] = std_scaler.fit_transform(X_train[col_to_scale])

# 4. ONLY Transform the TESTING data
X_test[col_to_scale] = std_scaler.transform(X_test[col_to_scale])

print("--- STANDARD SCALED TRAINING DATA---")
print(X_train)

### Exercise B: Min-Max Scaler

In [14]:
# 1. Create the scaler object
min_max_scaler = MinMaxScaler()

# 2. Tell the scaler which columns need to be scaled 
# (We don't want to scale our 1s and 0s from the City/Gender columns!)
col_to_scale = ['Age','Score']

# 2. Fit and Transform the TRAINING data
X_train[col_to_scale] = min_max_scaler.fit_transform(X_train[col_to_scale])

# 4. ONLY Transform the TESTING data
X_test[col_to_scale] = min_max_scaler.transform(X_test[col_to_scale])

print("--- STANDARD SCALED TRAINING DATA---")
print(X_train)

--- STANDARD SCALED TRAINING DATA---
    Gender_Female  Gender_Male  City_Bangalore  City_Chennai  City_Delhi  \
7             1.0          0.0             0.0           0.0         1.0   
10            0.0          1.0             0.0           1.0         0.0   
14            0.0          1.0             0.0           0.0         1.0   
6             0.0          1.0             1.0           0.0         0.0   

    City_Mumbai   Age     Score  
7           0.0  0.00  0.740741  
10          0.0  0.66  0.627451  
14          0.0  1.00  0.000000  
6           0.0  0.32  1.000000  
